# Übung 3 - Anomalie-Scores und Thresholds

Das Modelltraining ist unüberwacht: Für X_train gibt es keine verwendbaren Labels. Die Labels der Kalibrierungs- und Testdaten sind nur für die spätere Schwellenanalyse vorhanden.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

def normal_data(n, shift=(0, 0), scale=1.0):
    group = rng.choice(2, n, p=[.6, .4])
    means = np.array([[-2, 0], [2, 1]]) + np.asarray(shift)
    covs = np.array([
        [[.7, .3], [.3, .5]],
        [[.55, -.2], [-.2, .7]],
    ]) * scale**2
    return np.vstack([rng.multivariate_normal(means[g], covs[g]) for g in group])

def anomaly_data(n):
    obvious = rng.uniform([-5.5, -4], [5.5, 4], size=(n//2, 2))
    obvious = obvious[np.abs(obvious[:, 0]) > 3.8]
    while len(obvious) < n//2:
        more = rng.uniform([-5.5, -4], [5.5, 4], size=(n, 2))
        obvious = np.vstack([obvious, more[np.abs(more[:, 0]) > 3.8]])
    subtle = rng.multivariate_normal([0, .4], [[.9, 0], [0, .35]], size=n-n//2)
    return np.vstack([obvious[:n//2], subtle])

X_train = np.vstack([normal_data(1200), anomaly_data(12)])
X_cal = np.vstack([normal_data(450), anomaly_data(55)])
y_cal = np.r_[np.zeros(450, dtype=int), np.ones(55, dtype=int)]
X_test = np.vstack([normal_data(800), anomaly_data(80)])
y_test = np.r_[np.zeros(800, dtype=int), np.ones(80, dtype=int)]

cal_order = rng.permutation(len(X_cal))
test_order = rng.permutation(len(X_test))
X_cal, y_cal = X_cal[cal_order], y_cal[cal_order]
X_test, y_test = X_test[test_order], y_test[test_order]
print("Fit, Kalibrierung, Test:", X_train.shape, X_cal.shape, X_test.shape)


Fit, Kalibrierung, Test: (1212, 2) (505, 2) (880, 2)


## Aufgabe 1 - Isolation Forest als Score-Modell

1. Passe einen StandardScaler ausschließlich auf X_train an.
2. Trainiere einen IsolationForest mit mindestens 250 Bäumen und random_state=42.
3. Erzeuge Scores für Training, Kalibrierung und Test, wobei größere Werte auffälliger bedeuten.
4. Visualisiere die Test-Scores getrennt nach den nur zur Kontrolle bekannten Labels.


In [2]:
# TODO: Modell trainieren und konsistent orientierte Scores erzeugen


## Aufgabe 2 - Unüberwachte Quantilregeln

Nutze die Trainingsquantile 0.90, 0.95, 0.975, 0.99 und 0.995 als Thresholds. Berichte auf dem Testdatensatz jeweils:

- markierten Anteil
- Precision, Recall und F1
- False-Positive-Rate
- Zahl der übersehenen Anomalien

Erkläre, welche Annahme über die Anomalierate in einer Quantilregel steckt.


In [3]:
# TODO: Threshold-Tabelle berechnen


## Aufgabe 3 - Label- und kostenbasierte Kalibrierung

1. Wähle mit y_cal einen Threshold, der F1 maximiert.
2. Wähle einen zweiten Threshold, wenn ein False Positive 1 und ein False Negative 15 Kosteneinheiten verursacht.
3. Bewerte beide Schwellen genau einmal auf dem Testdatensatz.
4. Begründe, warum die Threshold-Wahl nun überwacht ist, obwohl der Isolation Forest unüberwacht trainiert wurde.


In [4]:
# TODO: F1- und Kosten-Threshold auf Kalibrierungsdaten wählen


## Aufgabe 4 - Distribution Shift

Erzeuge 1000 neue normale Punkte mit normal_data(1000, shift=(0.4, 0.2), scale=1.15). Wende den ursprünglichen 99-Prozent-Threshold unverändert an und bestimme die Fehlalarmrate.

Vergleiche sie mit 1000 normalen Punkten ohne Shift. Welche Komponenten eines produktiven Anomaliesystems müssten überwacht werden?


In [5]:
# TODO: Fehlalarmrate vor und nach Shift vergleichen
